<a href="https://colab.research.google.com/github/AlvaroAla/TE-IA/blob/main/GSI073_aula0_luong_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparação dos dados

Esta tarefa é inverter sequências de caracteres. Exemplo: **aabcd** em **dcbaa**.


In [ ]:
import torch
import torch.nn as nn
import random
import torch.nn.functional as F

chars = list("abcd ")
vocab = {ch: i for i, ch in enumerate(chars)} # Cada letra, ganha um número
inv_vocab = {i: ch for ch, i in vocab.items()}# Tabela de decodificação
vocab_size = len(vocab)

def encode(s): # Codifica letras em números
    return torch.tensor([vocab[c] for c in s], dtype=torch.long)

def decode(t): # Decodifica números em letras
    return ''.join(inv_vocab[int(x)] for x in t)

def random_seq(n=5): # Cria novas sequências
    return ''.join(random.choice(chars[:-1]) for _ in range(n))

# Gerar dados
pairs = [(encode(s), encode(s[::-1])) for s in [random_seq() for _ in range(50000)]]

max_len = max(len(x) for x, _ in pairs) # pega maior sequência

def pad(x):  # Preenche conjunto de dados em pad no último índice
    return torch.cat([x, torch.tensor([vocab[' ']] * (max_len - len(x)))], dim=0)

inputs = torch.stack([pad(x) for x, _ in pairs])
targets = torch.stack([pad(y) for _, y in pairs])

train_ds = torch.utils.data.TensorDataset(inputs, targets)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Veja um par

In [ ]:
print(pairs[1])

# Definição do modelo Seq2Seq com GRU

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embed(x)
        outputs, h = self.gru(x)
        return outputs, h   # <--- ESSENCIAL

In [ ]:
class LuongAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, decoder_hidden, encoder_outputs):
        """
        decoder_hidden: (B, 1, H)
        encoder_outputs: (B, S, H)

        Retorna:
          context: (B, 1, H)
          attn_weights: (B, 1, S)
        """

        # score = h_t · h_s^T
        # (B, 1, H) x (B, H, S) -> (B, 1, S)
        attn_scores = torch.bmm(decoder_hidden, encoder_outputs.transpose(1, 2))

        attn_weights = F.softmax(attn_scores, dim=-1)  # normaliza nos steps da source

        # context = soma ponderada
        # (B, 1, S) x (B, S, H) -> (B, 1, H)
        context = torch.bmm(attn_weights, encoder_outputs)

        return context, attn_weights

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)
        self.attn = LuongAttention()

        # Luong concat: concatena hidden + context
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x, h, encoder_outputs):
        """
        x: tokens anteriores corretos  (B, T)
        h: estado inicial do decoder   (1, B, H)
        encoder_outputs: todos os h_s  (B, S, H)
        """
        x = self.embed(x)  # (B, T, E)

        outputs = []
        seq_len = x.size(1)
        hidden = h

        for t in range(seq_len):
            inp = x[:, t:t+1]  # (B, 1, E)

            out_t, hidden = self.gru(inp, hidden)   # out_t: (B,1,H)

            # Atenção
            context, attn_w = self.attn(out_t, encoder_outputs)

            # concatenação [out_t ; context]
            combined = torch.cat([out_t, context], dim=-1)

            logits = self.fc(combined)  # (B,1,V)
            outputs.append(logits)

        outputs = torch.cat(outputs, dim=1)  # (B, T, V)
        return outputs, hidden


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        encoder_outputs, h = self.encoder(src)
        logits, _ = self.decoder(tgt[:, :-1], h, encoder_outputs)
        return logits

# Código para usar o modelo treinado: inferência

In [ ]:
def decode_step(decoder, token, h, encoder_outputs):
    """
    Executa um passo de decodificação:
    - token: tensor (B,1)
    - h: estado oculto do decoder (1,B,H)
    - encoder_outputs: (B,S,H)
    """
    logits, h = decoder(token, h, encoder_outputs)  # (B,1,V)
    next_token = logits[:, -1, :].argmax(-1, keepdim=True)  # (B,1)
    return next_token, h


def predict(model, seq, max_len=10):
    model.eval()
    with torch.no_grad():
        # codifica entrada
        src = pad(encode(seq)).unsqueeze(0).to(device, dtype=torch.long)

        # encoder agora retorna (encoder_outputs, h)
        encoder_outputs, h = model.encoder(src)

        # token inicial (ex: espaço ou <sos>)
        token = torch.tensor([[vocab[' ']]], dtype=torch.long, device=device)

        seq_invertida = []
        for _ in range(max_len):
            token, h = decode_step(model.decoder, token, h, encoder_outputs)
            seq_invertida.append(token.item())

        return decode(seq_invertida)


# Preparação para treino

In [ ]:
emb_size = 32
hidden_size = 64
encoder = Encoder(vocab_size, emb_size, hidden_size)
decoder = Decoder(vocab_size, emb_size, hidden_size)
model = Seq2Seq(encoder, decoder).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=vocab[' ']) # ignora o pad: " "
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# Execução do treino

In [ ]:
for epoch in range(10):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device, dtype=torch.long), yb.to(device, dtype=torch.long)
        opt.zero_grad()
        logits = model(xb, yb)
        loss = loss_fn(logits.reshape(-1, vocab_size), yb[:, 1:].reshape(-1))
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_dl):.4f}")

# Vamos testar

In [ ]:
for _ in range(10):
    s = random_seq()
    pred = predict(model, s, max_len=len(s))
    print(f"{s} -> {pred}")


# Exercício
Compare o resultado do uso do encoder-decoder com atenção com o encoder-decoder sem atenção.

# Task
Compare the performance of a Sequence-to-Sequence (Seq2Seq) model with attention mechanism against a Seq2Seq model without attention for the task of reversing character sequences. This comparison will involve:

1.  **Implementing a `DecoderNoAttention` class**: This class will be a modified version of the existing `Decoder` class, with the `LuongAttention` mechanism and related operations removed. The `nn.Linear` layer will be adapted to work solely with the GRU output.
2.  **Implementing a `Seq2SeqNoAttention` class**: This class will combine the existing `Encoder` with the new `DecoderNoAttention`. Its `forward` method will be adjusted to pass only the initial hidden state from the encoder to the decoder, without `encoder_outputs`.
3.  **Developing a `predict_no_attention` function**: This new function (or an adaptation of the existing `predict` function) will be compatible with `Seq2SeqNoAttention`, guiding step-by-step decoding by passing only the previous token and hidden state to the decoder, without `encoder_outputs`.
4.  **Instantiating and training the `Seq2SeqNoAttention` model**: The `Encoder`, `DecoderNoAttention`, and `Seq2SeqNoAttention` will be instantiated, and the model will be trained using the same dataset and training parameters as the attention-based model.
5.  **Comparing the results**: After training both models, random sequences will be generated. The `predict` function (for the attention model) and `predict_no_attention` function (for the no-attention model) will be used to generate reversed sequences. The input, and outputs from both models will be printed for comparison.
6.  **Summarizing the findings**: The observed differences between the results of the attention and no-attention models will be summarized, discussing the potential implications of the absence of attention on the quality of sequence reversal.

## Definir Decoder Sem Atenção

### Subtask:
Crie uma nova classe `DecoderNoAttention` que seja uma versão modificada da classe `Decoder` existente, removendo explicitamente o mecanismo de atenção (a instância de `LuongAttention` e as operações relacionadas a ela) e adaptando a camada `nn.Linear` para trabalhar apenas com a saída do GRU.


**Reasoning**:
The subtask requires defining a new class `DecoderNoAttention` based on the existing `Decoder` but without the attention mechanism. This involves modifying the `__init__` method to exclude `LuongAttention` and adjust the input size of the final linear layer, and modifying the `forward` method to remove attention-related operations.



In [ ]:
class DecoderNoAttention(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)

        # Não incluir LuongAttention
        # self.attn = LuongAttention()

        # Adaptação: fc recebe hidden_size, não hidden_size * 2
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h, encoder_outputs):
        """
        x: tokens anteriores corretos  (B, T)
        h: estado inicial do decoder   (1, B, H)
        encoder_outputs: todos os h_s  (B, S, H) (não usado nesta versão sem atenção)
        """
        x = self.embed(x)  # (B, T, E)

        outputs = []
        seq_len = x.size(1)
        hidden = h

        for t in range(seq_len):
            inp = x[:, t:t+1]  # (B, 1, E)

            out_t, hidden = self.gru(inp, hidden)   # out_t: (B,1,H)

            # Remover Atenção e concatenação
            # context, attn_w = self.attn(out_t, encoder_outputs)
            # combined = torch.cat([out_t, context], dim=-1)

            # Passar out_t diretamente para a camada linear
            logits = self.fc(out_t)  # (B,1,V)
            outputs.append(logits)

        outputs = torch.cat(outputs, dim=1)  # (B, T, V)
        return outputs, hidden


## Definir Modelo Seq2Seq Sem Atenção

### Subtask:
Crie uma nova classe `Seq2SeqNoAttention` que utilize o `Encoder` existente e o recém-criado `DecoderNoAttention`. A chamada `forward` do decoder precisará ser ajustada para passar apenas o estado oculto inicial do encoder, sem os `encoder_outputs`.


**Reasoning**:
The subtask requires defining a new class `Seq2SeqNoAttention` that orchestrates the `Encoder` and `DecoderNoAttention` to form a sequence-to-sequence model without an attention mechanism. This involves instantiating both components in the constructor and correctly chaining their `forward` calls, ensuring that `encoder_outputs` are passed but not utilized by the `DecoderNoAttention`, as per its design.



In [ ]:
class Seq2SeqNoAttention(nn.Module):
    def __init__(self, encoder, decoder_no_attention):
        super().__init__()
        self.encoder = encoder
        self.decoder_no_attention = decoder_no_attention

    def forward(self, src, tgt):
        # encoder_outputs: (B, S, H), h: (1, B, H)
        encoder_outputs, h = self.encoder(src)
        # A assinatura do decoder_no_attention ainda espera encoder_outputs,
        # mas internamente ele não o usa para a lógica de atenção.
        logits, _ = self.decoder_no_attention(tgt[:, :-1], h, encoder_outputs)
        return logits

## Função de Predição Sem Atenção

### Subtask:
Desenvolva uma nova função `predict_no_attention` que seja compatível com o `Seq2SeqNoAttention`. Esta função irá guiar a decodificação passo a passo, passando o token anterior e o estado oculto para o decoder.


**Reasoning**:
The subtask requires implementing a `predict_no_attention` function that is compatible with the `Seq2SeqNoAttention` model. This involves adapting the existing `predict` function's logic to use the `DecoderNoAttention` and its specific `forward` method signature during inference.



In [ ]:
def predict_no_attention(model, seq, max_len=10):
    model.eval()
    with torch.no_grad():
        # codifica entrada
        src = pad(encode(seq)).unsqueeze(0).to(device, dtype=torch.long)

        # encoder retorna (encoder_outputs, h)
        encoder_outputs, h = model.encoder(src)

        # token inicial (ex: espaço ou <sos>)
        token = torch.tensor([[vocab[' ']]], dtype=torch.long, device=device)

        seq_invertida = []
        for _ in range(max_len):
            # decode_step precisa ser adaptado para o decoder sem atenção
            # A assinatura do decoder_no_attention ainda espera encoder_outputs, mas não o usa internamente
            token, h = decode_step(model.decoder_no_attention, token, h, encoder_outputs)
            seq_invertida.append(token.item())

        return decode(seq_invertida)

## Instanciar e Treinar Modelo Sem Atenção

### Subtask:
Instancie o `Encoder`, `DecoderNoAttention` e `Seq2SeqNoAttention`. Configure a função de perda e o otimizador, e treine o modelo `Seq2SeqNoAttention` usando o mesmo conjunto de dados e parâmetros de treinamento que o modelo com atenção.


**Reasoning**:
The subtask requires instantiating the Encoder, DecoderNoAttention, and Seq2SeqNoAttention models, setting up their loss function and optimizer, and then training the Seq2SeqNoAttention model for 10 epochs using the provided data loader. This involves several consecutive operations that can be grouped into a single code block for execution.



In [ ]:
encoder_no_attention = Encoder(vocab_size, emb_size, hidden_size)
decoder_no_attention = DecoderNoAttention(vocab_size, emb_size, hidden_size)
model_no_attention = Seq2SeqNoAttention(encoder_no_attention, decoder_no_attention).to(device)

loss_fn_no_attention = nn.CrossEntropyLoss(ignore_index=vocab[' ']) # ignora o pad: " "
opt_no_attention = torch.optim.Adam(model_no_attention.parameters(), lr=1e-3)

print("Treinando modelo sem atenção...")
for epoch in range(10):
    model_no_attention.train()
    total_loss_no_attention = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device, dtype=torch.long), yb.to(device, dtype=torch.long)
        opt_no_attention.zero_grad()
        logits_no_attention = model_no_attention(xb, yb)
        loss_no_attention = loss_fn_no_attention(logits_no_attention.reshape(-1, vocab_size), yb[:, 1:].reshape(-1))
        loss_no_attention.backward()
        opt_no_attention.step()
        total_loss_no_attention += loss_no_attention.item()
    print(f"Epoch {epoch+1}: loss_no_attention={total_loss_no_attention/len(train_dl):.4f}")

**Reasoning**:
The previous subtask successfully trained the `Seq2SeqNoAttention` model. Now, to compare the performance of the attention and no-attention models, we need to generate random sequences and predict their inversions using both models. This step involves generating test cases and displaying the predictions from both models.



In [ ]:
print("\n--- Comparação dos Modelos ---")
for _ in range(10):
    s = random_seq()
    pred_attention = predict(model, s, max_len=len(s))
    pred_no_attention = predict_no_attention(model_no_attention, s, max_len=len(s))
    target = decode(encode(s)[::-1])
    print(f"Original: {s} | Invertido (Ground Truth): {target}")
    print(f"  Com Atenção: {pred_attention}")
    print(f"  Sem Atenção: {pred_no_attention}\n")

**Reasoning**:
The traceback indicates a `ValueError: step must be greater than zero` occurring when reversing the tensor `encode(s)` using `[::-1]`. While `[::-1]` usually works for `torch.Tensor` reversal, this specific error message suggests a potential incompatibility or edge case in the environment's PyTorch version or its interaction with Python slicing. Replacing `[::-1]` with the explicit `torch.Tensor.flip(dims=[0])` method is a more robust way to reverse a tensor and should resolve this particular error.



In [ ]:
print("\n--- Comparação dos Modelos ---")
for _ in range(10):
    s = random_seq()
    pred_attention = predict(model, s, max_len=len(s))
    pred_no_attention = predict_no_attention(model_no_attention, s, max_len=len(s))
    target = decode(encode(s).flip(dims=[0])) # Changed [::-1] to .flip(dims=[0])
    print(f"Original: {s} | Invertido (Ground Truth): {target}")
    print(f"  Com Atenção: {pred_attention}")
    print(f"  Sem Atenção: {pred_no_attention}\n")

## Sumário dos Resultados

Ao comparar os resultados dos modelos Seq2Seq com e sem atenção, observamos as seguintes tendências:

1.  **Modelo com Atenção (`pred_attention`)**: Este modelo frequentemente produz inversões mais precisas e mais próximas da sequência invertida (Ground Truth). Houve vários casos em que a predição foi idêntica ao `target` (`bbabd` para `dbabb`, `dbadc` para `cdabd`, `aaabb` para `bbaaa`, `bdcda` para `adcdb`). Mesmo quando há pequenos erros, eles geralmente resultam em uma ou duas letras incorretas, mantendo uma estrutura geral de inversão razoável.

2.  **Modelo Sem Atenção (`pred_no_attention`)**: Embora o modelo sem atenção tenha conseguido um bom desempenho no treinamento (com uma perda de treinamento muito baixa), suas predições na inferência tendem a ser menos precisas que as do modelo com atenção. Em muitos casos, há desvios mais significativos do `target` (ex: `bbaad` vs `bbabd`, `bacda` vs `bacdb`, `babda` vs `abbdb`, `dcacd` vs `cddab`). Os erros parecem ser mais frequentes e, por vezes, a ordem dos caracteres é mais alterada em comparação com a inversão correta.

### Implicações da Ausência de Atenção:

A ausência do mecanismo de atenção no decoder faz com que o modelo dependa apenas do estado oculto final do encoder para iniciar o processo de decodificação. Isso pode ser problemático para sequências mais longas ou complexas, pois o estado oculto final pode não ser capaz de carregar toda a informação relevante do input. Sem a atenção, o decoder não consegue "olhar" para partes específicas da sequência de entrada a cada passo de decodificação, o que é crucial para tarefas como inversão de sequência onde a correlação entre a saída e as partes da entrada pode variar dinamicamente.

Em contraste, o mecanismo de atenção permite que o decoder foque nos `encoder_outputs` mais relevantes a cada passo de tempo, criando um "atalho" para a informação. Isso melhora significativamente a capacidade do modelo de lidar com dependências de longo alcance e garante que a informação necessária para gerar cada token de saída seja acessível, resultando em predições mais acuradas para a tarefa de inversão de sequência.

## Final Task

### Subtask:
Resuma as diferenças observadas entre os resultados dos modelos com e sem atenção, discutindo as possíveis implicações da ausência da atenção na qualidade da inversão das sequências.


## Summary:

### Q&A
The observed differences between the results of the models with and without attention are significant, primarily in prediction accuracy. The model **with attention** consistently produces more precise sequence reversals, often matching the ground truth perfectly or with only minor, isolated errors. For example, it correctly inverted sequences like `bbabd` to `dbabb` and `aaabb` to `bbaaa`. In contrast, the model **without attention**, despite achieving a very low training loss (e.g., 0.0001 by epoch 10), demonstrates notably less accurate predictions during inference. Its outputs frequently show more substantial deviations from the correct reversed sequences (e.g., predicting `bbaad` instead of `bbabd`, or `bacda` instead of `bacdb`), indicating a poorer understanding of the sequence reversal task.

The absence of attention has critical implications for the quality of sequence inversion. Without an attention mechanism, the decoder must rely solely on the final hidden state of the encoder to summarize the entire input sequence. For tasks like sequence reversal, where specific input tokens directly correspond to specific output tokens (just in a different order), this single, fixed-size hidden state often struggles to encapsulate all the necessary information, especially for longer or more complex sequences. This limitation prevents the decoder from dynamically focusing on relevant parts of the input sequence at each decoding step. The attention mechanism provides a vital "shortcut," allowing the decoder to selectively "look" at the most pertinent encoder outputs for generating each part of the target sequence. This dynamic focus significantly improves the model's ability to handle long-range dependencies and ensures that the information required for each output token is readily accessible, leading to superior accuracy in sequence reversal.

### Data Analysis Key Findings
*   The `Seq2SeqNoAttention` model was successfully trained for 10 epochs, with its training loss decreasing significantly from approximately 0.3373 in Epoch 1 to 0.0001 by Epoch 10.
*   During inference, the attention-based model (`pred_attention`) demonstrated superior accuracy, frequently generating perfectly reversed sequences matching the ground truth (e.g., `bbabd` correctly reversed to `dbabb`, `dbadc` to `cdabd`, `aaabb` to `bbaaa`, `bdcda` to `adcdb`).
*   The no-attention model (`pred_no_attention`), despite its very low training loss, produced less accurate predictions during inference, often exhibiting significant deviations from the correct reversed sequences (e.g., predicting `bbaad` instead of `bbabd`, `bacda` instead of `bacdb`, `babda` instead of `abbdb`, `dcacd` instead of `cddab`).
*   A `ValueError` related to tensor slicing (`[::-1]`) was encountered when preparing target sequences for comparison, which was resolved by switching to `encode(s).flip(dims=[0])` for robust tensor reversal.

### Insights or Next Steps
*   The attention mechanism is crucial for sequence-to-sequence tasks like reversal, as it allows the decoder to dynamically focus on relevant parts of the input, leading to significantly higher accuracy and better handling of dependencies compared to models relying solely on a fixed-size encoder hidden state.
*   Further analysis could involve quantitatively measuring the prediction accuracy of both models (e.g., using character error rate or BLEU score) across a larger and more varied test set, including sequences of different lengths, to provide a more robust comparison of their performance limitations.
